In [ ]:
## Exploration of MTA GTFS Live Feed for checking content of the feed 
## and testing out Kafka producer code to send the feed data to a Kafka topic
## N.B. Code is Vibe-coded

import requests
from kafka import KafkaProducer
import json

FEED_URL = "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-ace"
KAFKA_BROKER = "localhost:9092"
KAFKA_TOPIC = "mta-gtfs"

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

In [13]:
response = requests.get(FEED_URL)

print("Content-Type:", response.headers.get('Content-Type'))
print("Status Code:", response.status_code)

print("First 100 bytes:", response.content[:100])

Content-Type: text/plain
Status Code: 200
First 100 bytes: b'\nG\n\x031.0\x18\xd7\xe9\xdc\xcc\x06\xca>9\n\x031.0\x12\x0b\n\x01A\x12\x06\x10\xe7\x85\xdd\xcc\x06\x12\x0b\n\x01C\x12\x06\x10\xe7\x85\xdd\xcc\x06\x12\x0b\n\x01E\x12\x06\x10\xe7\x85\xdd\xcc\x06\x12\x0b\n\x01H\x12\x06\x10\xe6\x85\xdd\xcc\x06\x12J\n\x07000001A\x1a?\n=\n\x0e052300_A..'


In [ ]:
## Existing package for parsing GTFS Realtime feeds 

from google.transit import gtfs_realtime_pb2

response = requests.get(FEED_URL)

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(response.content)

# Explore the schema
print("Feed Header:")
print(f"  - GtfsRealtimeVersion: {feed.header.gtfs_realtime_version}")
print(f"  - Timestamp: {feed.header.timestamp}")
print(f"  - Number of entities: {len(feed.entity)}")

# Print first entity to see the structure
if feed.entity:
    entity = feed.entity[0]
    print("\nFirst Entity:")
    print(f"  - ID: {entity.id}")
    
    for field, value in entity.ListFields():
        print(f"  - {field.name}: {type(value).__name__}")
    
    if entity.HasField('trip_update'):
        trip_update = entity.trip_update
        for field, value in trip_update.ListFields():
            print(f"    - {field.name}: {type(value).__name__}")
    
# Details of the Schema as per : https://gtfs.org/documentation/realtime/feed-entities/trip-updates/


Feed Header:
  - GtfsRealtimeVersion: 1.0
  - Timestamp: 1771517143
  - Number of entities: 256

First Entity:
  - ID: 000001A
  - id: str
  - trip_update: TripUpdate
    - trip: TripDescriptor


In [ ]:
## Sending out the feed data to a Kafka topic
## Kafka instance is running locally through a Docker container

if feed.entity:
    entity = feed.entity[0]
    
    if entity.HasField('trip_update'):
        trip_update = entity.trip_update
        
        event = {
            'entity_id': entity.id,
            'trip_id': trip_update.trip.trip_id,
            'route_id': trip_update.trip.route_id,
            'timestamp': feed.header.timestamp,
            'stop_time_updates': [
                {
                    'stop_sequence': stop.stop_sequence,
                    'arrival_time': stop.arrival.time if stop.HasField('arrival') else None,
                    'departure_time': stop.departure.time if stop.HasField('departure') else None,
                }
                for stop in trip_update.stop_time_update
            ]
        }
        
        producer.send(KAFKA_TOPIC, value=event)
        producer.flush()
        print(f"✓ Sent event to Kafka topic '{KAFKA_TOPIC}'")
        print(f"  Trip ID: {event['trip_id']}, Route ID: {event['route_id']}")

✓ Sent event to Kafka topic 'mta-gtfs'
  Trip ID: 052300_A..N55R, Route ID: A
